In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# 1. Load processed matrices
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Scripts" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "Processed"

X_circ_logcpm = pd.read_csv(PROCESSED_DIR / "GSE236932_circRNA_log2cpm.csv", index_col=0)
mrna_log2 = pd.read_csv(PROCESSED_DIR / "GSE236932_mRNA_log2.csv", index_col=0)
scores_df = pd.read_csv(PROCESSED_DIR / "GSE236932_target_scores.csv", index_col=0)

sample_cols = scores_df.index.tolist()
X_circ_logcpm = X_circ_logcpm[sample_cols]
mrna_log2 = mrna_log2[sample_cols]

# Filter for expressed mRNAs (log2 expression > 1 in at least 20 samples) to avoid zero-inflation noise
expressed_mrnas = mrna_log2.loc[(mrna_log2 > 1.0).sum(axis=1) >= 20]
print(f"Screening {expressed_mrnas.shape[0]} expressed mRNAs for direct ceRNA co-expression...")

# 2. Vectorized Spearman correlation across all expressed mRNAs
def screen_cerna_partners(circ_id, circ_label, top_n=15):
    c_vec = X_circ_logcpm.loc[circ_id].values
    hyp_vec = scores_df["Hypoxia_Buffa"].values
    
    # Rank-transform for ultra-fast vectorized Spearman rho
    c_rank = pd.Series(c_vec).rank().values
    c_rank_centered = c_rank - c_rank.mean()
    
    m_ranks = expressed_mrnas.rank(axis=1).values
    m_ranks_centered = m_ranks - m_ranks.mean(axis=1, keepdims=True)
    
    num = np.dot(m_ranks_centered, c_rank_centered)
    den = np.sqrt(np.sum(m_ranks_centered**2, axis=1) * np.sum(c_rank_centered**2))
    rhos = num / den
    
    # Compute exact p-values & BH FDR q-values for top candidates
    pvals = [spearmanr(c_vec, expressed_mrnas.iloc[i].values)[1] for i in range(len(expressed_mrnas))]
    _, qvals, _, _ = multipletests(pvals, method="fdr_bh")
    
    res = pd.DataFrame({
        "mRNA_Gene": expressed_mrnas.index,
        "rho_with_circ": rhos,
        "FDR_qval": qvals
    }).set_index("mRNA_Gene")
    
    # Also compute each mRNA's correlation with Hypoxia_Buffa to trace the 2-step cascade
    hyp_rhos = [spearmanr(expressed_mrnas.loc[g].values, hyp_vec)[0] for g in res.index]
    res["rho_with_Hypoxia"] = hyp_rhos
    
    # Filter for significant positive ceRNA partners (rho > 0.5, FDR < 0.01) that suppress hypoxia (rho_with_Hypoxia < -0.3)
    cascade_hits = res[(res["rho_with_circ"] > 0.5) & (res["FDR_qval"] < 0.01) & (res["rho_with_Hypoxia"] < -0.3)]
    cascade_hits = cascade_hits.sort_values(by="rho_with_circ", ascending=False)
    
    print(f"\n=== Top Direct ceRNA Partner mRNAs for {circ_label} ({circ_id}) ===")
    print(f"Total significant positive partners (rho > 0.5, FDR < 0.01): {(res['rho_with_circ'] > 0.5).sum()}")
    print(cascade_hits.head(top_n).round(4).to_string())
    return cascade_hits

lifr_partners = screen_cerna_partners("chr5:38523419|38530666", "circLIFR (hsa_circ_0072309)")
l3mbtl4_partners = screen_cerna_partners("chr18:6213149|6312056", "circL3MBTL4")

lifr_partners.to_csv(PROCESSED_DIR / "GSE236932_circLIFR_ceRNA_partners.csv")
l3mbtl4_partners.to_csv(PROCESSED_DIR / "GSE236932_circL3MBTL4_ceRNA_partners.csv")

Screening 32514 expressed mRNAs for direct ceRNA co-expression...

=== Top Direct ceRNA Partner mRNAs for circLIFR (hsa_circ_0072309) (chr5:38523419|38530666) ===
Total significant positive partners (rho > 0.5, FDR < 0.01): 43
            rho_with_circ  FDR_qval  rho_with_Hypoxia
mRNA_Gene                                            
LIFR-AS1           0.6085    0.0020           -0.7062
HIF3A              0.5898    0.0030           -0.6634
RAB3IL1            0.5768    0.0036           -0.5242
PZP                0.5606    0.0036           -0.5671
TPSB2              0.5577    0.0036           -0.5559
GAS1               0.5501    0.0036           -0.5853
MFAP4              0.5497    0.0036           -0.5543
SCARA5             0.5420    0.0036           -0.5649
GPIHBP1            0.5416    0.0036           -0.5081
TPSAB1             0.5405    0.0036           -0.4649
TNXB               0.5392    0.0036           -0.5823
HAND2-AS1          0.5341    0.0037           -0.6402
AL606804.1       

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 11.5 MB/s eta 0:00:00 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
